<a href="https://colab.research.google.com/github/samsarkar184/Self-Supervised-Learning/blob/main/SimCLR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#imports
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader,Subset
from torchvision import transforms,datasets
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [ ]:
#gpu check
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
#transformation of image
train_transform=transforms.Compose([transforms.RandomHorizontalFlip(),
                                    transforms.RandomResizedCrop(32,scale=(0.5,1.0)),
                                    transforms.RandomApply([
                                        transforms.ColorJitter(brightness=0.8,contrast=0.8,saturation=0.8,hue=0.2)
                                    ],p=0.8),
                                    transforms.ToTensor()])

In [29]:
#import CIFAR-10 train dataset
train_dataset=datasets.CIFAR10(root="./data",
                               train=True,
                               download=True,
                               transform=None)

In [30]:
#import CIFAR-10 test dataset
transform=transforms.ToTensor()
test_dataset=datasets.CIFAR10(root="./data",
                              train=False,
                              download=True,
                              transform=transform)

In [31]:
#take a subset of the dataset for training and testing
train_subset=Subset(train_dataset,range(5000))
test_subset=Subset(test_dataset,range(1000))
train_loader=DataLoader(train_subset,batch_size=128,shuffle=True)
test_loader=DataLoader(test_subset,batch_size=128,shuffle=True)

In [32]:
# creating SimClr wrapper
from torch.utils.data import Dataset
class Simdataset(Dataset):
    def __init__(self,dataset,transform):
        self.dataset = dataset
        self.transform = transform
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, index):
        image = self.dataset[index][0]
        view_1 = self.transform(image)
        view_2 = self.transform(image)
        return view_1, view_2

In [33]:
# creating object of simdataset
sim_dataset = Simdataset(train_subset, train_transform)

In [34]:
# displaying the no. of samples and two augmented view
print("Number of samples:", len(sim_dataset))

view_1, view_2 = sim_dataset[0]

print("View 1 shape:", view_1.shape)
print("View 2 shape:", view_2.shape)

Number of samples: 5000
View 1 shape: torch.Size([3, 32, 32])
View 2 shape: torch.Size([3, 32, 32])


In [35]:
# creating SimClr DataLoader
from torch.utils.data import DataLoader
sim_loader = DataLoader(
    sim_dataset,
    batch_size = 128,
    shuffle = True,
    drop_last = True
)

In [36]:
# Testing the DataLoader
view_1_batch, view_2_batch = next(iter(sim_loader))
print("View 1 batch shape:", view_1_batch.shape)
print("View 2 batch shape:", view_2_batch.shape)

View 1 batch shape: torch.Size([128, 3, 32, 32])
View 2 batch shape: torch.Size([128, 3, 32, 32])


In [37]:
# build encoder
class Encoder(nn.Module):
  def __init__(self):
    super(Encoder,self).__init__()
    self.conv1=nn.Conv2d(3,32,kernel_size=3,padding=1)
    self.conv2=nn.Conv2d(32,64,kernel_size=3,padding=1)
    self.pool=nn.MaxPool2d(2,2)
    self.fc=nn.Linear(64*8*8,128)

  def forward(self,x):
    x=F.relu(self.conv1(x))
    x=self.pool(x)
    x=F.relu(self.conv2(x))
    x=self.pool(x)
    x=torch.flatten(x,1)
    x=self.fc(x)
    return x


In [38]:
#create Encoder object
encoder = Encoder()
encoder = encoder.to(device)

In [39]:
# checking encoder work
x = view_1_batch.to(device)
features = encoder(x)
print("Encoder output shape:", features.shape)

Encoder output shape: torch.Size([128, 128])


In [42]:
# build projection head
class ProjectionHead(nn.Module):
  def __init__(self):
    super(ProjectionHead, self).__init__()
    self.fc1 = nn.Linear(128, 64)
    self.fc2 = nn.Linear(64, 32)
  def forward(self, x):
    x = F.ReLU(self.fc1(x))
    x = self.fc2(x)
    return x


In [43]:
# create projection head object and testing the encoder and projection head work
projection_head = ProjectionHead()
features = encoder(view_1_batch)
projections = projection_head(features)
print("Encoder output shape:",features.shape)
print("Projection output shape:",projections.shape)

RuntimeError: Input type (torch.FloatTensor) and weight type (torch.cuda.FloatTensor) should be the same or input should be a MKLDNN tensor and weight is a dense tensor

In [ ]:
# processing both augmented views through encoder and projection head
features_1 = encoder(view_1_batch)
features_2 = encoder(view_2_batch)
projections_1 = projection_head(features_1)
projections_2 = projection_head(features_2)
print("Projection 1 shape:"projections_1.shape)
print("Projection 2 shape:",projections_2.shape)

In [ ]:
def nt_xent_loss(z1,z2,temperature=0.5):
  batch_size=z1.shape[0]

  z1=F.normalize(z1,dim=1)
  z2=F.normalize(z2,dim=1)

  z=torch.cat([z1,z2],dim=0)

  similarity=torch.matmul(z,z.T)
  similarity=similarity/temperature

  mask=torch.eye(2*batch_size,dtype=torch.bool,device=z.device)
  similarity=similarity.masked_fill(mask,-float('inf'))

  positive_indices=torch.cat([torch.arrange(batch_size,2*torch_size),
                              torch.arrange(0,batch_size)]).to(z.device)
  loss=F.cross_entropy(similarity,positive_indices)
  return loss